# 3. Text Analysis: Lyrics-Based Exploration of Audio Communities

This notebook extends the network analysis by using lyrics to **semantically interpret** the audio-based genre communities. We will:

1. **Recreate audio-based communities** (from notebook 2)
2. **Randomly sample genres** from each community (filtering out instrumental genres)
3. **Scrape lyrics** using the Genius API
4. **Preprocess text** and compute TF-IDF representations
5. **Generate word clouds** to explore thematic patterns within audio communities
6. **Analyze sentiment** and visualize thematic differences

**Key question:** Do genres that *sound* similar also *talk about* similar things? We use word clouds as a semantic interpretation layer for the audio communities.

## 0. Setup & Configuration

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import time
import re
import os
import nltk
import random

# Network analysis
import networkx as nx
from networkx.algorithms.community import louvain_communities, modularity

# ML / NLP
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Genius API
from lyricsgenius import Genius

# Word clouds
from wordcloud import WordCloud

# Plotting settings
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# ============================================================================
# CONFIGURATION - ADJUST THESE PARAMETERS AS NEEDED
# ============================================================================

# Paths
DATA_PATH = "data/clean_spotify_tracks_dataset.csv"  # cleaned dataset
ENV_PATH = "../.env.local"  # Path to .env file with Genius API token

# Random seed for reproducibility
RANDOM_SEED = 42

# Sampling parameters
N_GENRES_PER_COMMUNITY = 10  # Randomly select N genres from each community
SONGS_PER_GENRE = 1  # Number of songs to randomly sample per genre

# Genre filtering thresholds
MIN_TRACKS_PER_GENRE = 100  # Exclude genres with fewer tracks
MAX_INSTRUMENTALNESS = 0.5  # Exclude genres with instrumentalness >= this value

# Language filtering
ENGLISH_ONLY = True

# Community detection parameters (same as notebook 2 for reproducibility)
SIM_THRESHOLD = 0.3
LOUVAIN_SEED = 42

# Word cloud parameters
WORDCLOUD_MAX_WORDS = 50  # Maximum words in word clouds

# Output directories
OUTPUT_DIR = Path("data") / "text_analysis_output"
LYRICS_DIR = OUTPUT_DIR / "lyrics_by_community"
SAMPLES_DIR = OUTPUT_DIR / "song_samples"

# Create output directories
LYRICS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

# Set random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Configuration:")
print(f"  - Random seed: {RANDOM_SEED}")
print(f"  - {N_GENRES_PER_COMMUNITY} random genres per community")
print(f"  - {SONGS_PER_GENRE} random songs per genre")
print(f"  - Min tracks per genre: {MIN_TRACKS_PER_GENRE}")
print(f"  - Max instrumentalness: {MAX_INSTRUMENTALNESS}")
print(f"  - English only: {ENGLISH_ONLY}")
print(f"  - Output: {OUTPUT_DIR}")

Configuration:
  - Random seed: 42
  - 10 random genres per community
  - 1 random songs per genre
  - Min tracks per genre: 100
  - Max instrumentalness: 0.5
  - English only: True
  - Output: data/text_analysis_output


In [3]:
# ============================================================================
# GENRE EXCLUSION LIST FOR TEXT ANALYSIS
# These genres are excluded from TF-IDF/word cloud analysis because they are
# primarily non-English, making cross-genre text comparison unfair.
# Modify this list as needed.
# ============================================================================

EXCLUDED_GENRES = {
    # Pure nationality/language genres
    'brazil',
    'british',
    'french',
    'german',
    'indian',
    'iranian',
    'malay',
    'spanish',
    'swedish',
    'turkish',
    
    # Portuguese-dominant genres
    'forro',
    'mpb',
    'pagode',
    'samba',
    'sertanejo',
    
    # Asian-language genres
    'cantopop',
    'mandopop',
    'j-dance',
    'j-idol',
    'j-pop',
    'j-rock',
    'k-pop',
    
    # Spanish-dominant genres
    'latin',
    'latino',
    'reggaeton',
    'salsa',
    'tango',
    
    # Catch-all
    'world-music',
}

print(f"Excluding {len(EXCLUDED_GENRES)} non-English genres from text analysis:")
print(f"  {sorted(EXCLUDED_GENRES)}")

Excluding 28 non-English genres from text analysis:
  ['brazil', 'british', 'cantopop', 'forro', 'french', 'german', 'indian', 'iranian', 'j-dance', 'j-idol', 'j-pop', 'j-rock', 'k-pop', 'latin', 'latino', 'malay', 'mandopop', 'mpb', 'pagode', 'reggaeton', 'salsa', 'samba', 'sertanejo', 'spanish', 'swedish', 'tango', 'turkish', 'world-music']


In [4]:
# ============================================================================
# ARCHETYPAL GENRES FOR GENRE-LEVEL WORD CLOUDS
# These are well-known genres that span different audio communities.
# Modify this list as needed.
# ============================================================================

ARCHETYPAL_GENRES = [
    # Mainstream
    'pop',
    'rock',
    'hip-hop',
    'r-n-b',
    'country',
    
    # Electronic
    'edm',
    'house',
    'techno',
    
    # Heavy
    'metal',
    'hard-rock',
    'punk',
    
    # Mellow
    'jazz',
    'classical',
    'folk',
    'blues',
    'indie',
    
    # Other
    'reggae',
    'soul',
]

print(f"Archetypal genres for word clouds ({len(ARCHETYPAL_GENRES)}):")
print(f"  {ARCHETYPAL_GENRES}")

Archetypal genres for word clouds (18):
  ['pop', 'rock', 'hip-hop', 'r-n-b', 'country', 'edm', 'house', 'techno', 'metal', 'hard-rock', 'punk', 'jazz', 'classical', 'folk', 'blues', 'indie', 'reggae', 'soul']


In [5]:
# ============================================================================
# COMMON THEMES REMOVAL
# Some words are thematically meaningful but appear across ALL communities,
# making them non-differentiating. Toggle these to remove from word clouds.
# Set value to True to REMOVE the word, False to KEEP it.
# ============================================================================

REMOVE_COMMON_THEMES = {
    'love': False,      # Appears in most communities, toggle to see difference
    'heart': False,     # Often co-occurs with love
    'life': False,      # Very generic
    'world': False,     # Very generic
}

# Show current settings
print("Common theme removal settings:")
for word, remove in REMOVE_COMMON_THEMES.items():
    status = "REMOVED" if remove else "KEPT"
    print(f"  '{word}': {status}")

Common theme removal settings:
  'love': KEPT
  'heart': KEPT
  'life': KEPT
  'world': KEPT


In [6]:
# ============================================================================
# CUSTOM STOPWORDS FOR LYRICS
# These words are too generic/frequent to provide thematic insight.
# Combined with NLTK's English stopwords during preprocessing.
# ============================================================================

CUSTOM_STOPWORDS = {
    # === Slang & Contractions ===
    'wanna', 'gonna', 'gotta', 'finna', 'cause', 'cuz', 'bout', 'em', 
    'da', 'lil', 'ya', 'yo', 'til', 'aint', 'yall',
    
    # === High-Frequency Generic Verbs ===
    'got', 'get', 'gets', 'getting', 'gotten',
    'go', 'goes', 'going', 'gone', 'went',
    'let', 'lets', 'letting',
    'take', 'takes', 'taking', 'took', 'taken',
    'make', 'makes', 'making', 'made',
    'say', 'says', 'said', 'saying',
    'see', 'sees', 'saw', 'seen', 'seeing',
    'know', 'knows', 'knew', 'known', 'knowing',
    'think', 'thinks', 'thought', 'thinking',
    'tell', 'tells', 'told', 'telling',
    'look', 'looks', 'looked', 'looking',
    'come', 'comes', 'came', 'coming',
    'give', 'gives', 'gave', 'given', 'giving',
    'need', 'needs', 'needed', 'needing',
    'want', 'wants', 'wanted', 'wanting',
    'feel', 'feels', 'felt', 'feeling',
    'keep', 'keeps', 'kept', 'keeping',
    'stay', 'stays', 'stayed', 'staying',
    'turn', 'turns', 'turned', 'turning',
    'try', 'tries', 'tried', 'trying',
    'leave', 'leaves', 'left', 'leaving',
    'hold', 'holds', 'held', 'holding',
    'call', 'calls', 'called', 'calling',
    'fall', 'falls', 'fell', 'fallen', 'falling',
    'put', 'puts', 'putting',
    'run', 'runs', 'ran', 'running',
    'find', 'finds', 'found', 'finding',
    'live', 'lives', 'lived', 'living',
    'bring', 'brings', 'brought', 'bringing',
    'start', 'starts', 'started', 'starting',
    'move', 'moves', 'moved', 'moving',
    'walk', 'walks', 'walked', 'walking',
    'stand', 'stands', 'stood', 'standing',
    'hear', 'hears', 'heard', 'hearing',
    'play', 'plays', 'played', 'playing',
    'talk', 'talks', 'talked', 'talking',
    'watch', 'watches', 'watched', 'watching',
    'wait', 'waits', 'waited', 'waiting',
    'stop', 'stops', 'stopped', 'stopping',
    'lose', 'loses', 'lost', 'losing',
    'change', 'changes', 'changed', 'changing',
    'set', 'sets', 'setting',
    
    # === Modal/Auxiliary Verbs ===
    'could', 'would', 'should', 'might', 'must',
    'never', 'ever', 'always', 'still', 'already',
    
    # === Generic Nouns ===
    'back', 'away', 'right', 'left',
    'way', 'ways',
    'time', 'times',
    'thing', 'things', 'something', 'nothing', 'everything', 'anything',
    'man', 'men', 'woman', 'women',
    'girl', 'girls', 'boy', 'boys', 'guy', 'guys',
    'baby', 'babe',
    'night', 'nights', 'day', 'days', 'tonight', 'today',
    'mind', 'minds',
    'head', 'heads',
    'eye', 'eyes',
    'hand', 'hands',
    'place', 'places',
    'side', 'sides',
    'end', 'ends',
    'part', 'parts',
    'word', 'words',
    'name', 'names',
    'face', 'faces',
    'door', 'doors',
    'room', 'rooms',
    'home',
    'body', 'bodies',
    
    # === Generic Adjectives ===
    'little', 'big', 'small', 'large',
    'good', 'better', 'best',
    'bad', 'worse', 'worst',
    'real', 'really',
    'new', 'old',
    'long', 'short',
    'high', 'low',
    'hard', 'soft',
    'true', 'right', 'wrong',
    'whole', 'same', 'different',
    'last', 'first', 'next',
    'alone',
    
    # === Quantifiers & Pronouns ===
    'every', 'anymore', 'someone', 'everyone', 'anyone', 'somebody',
    'one', 'ones', 'two', 'three',
    
    # === Filler Words / Vocalizations ===
    'yeah', 'yea', 'yeh',
    'oh', 'ooh', 'oooh', 'ah', 'ahh', 'uh', 'uhh',
    'la', 'na', 'nah',
    'hey', 'hi', 'hello',
    'whoa', 'woah', 'wow',
    'hmm', 'mmm', 'mm',
    'ayy', 'aye', 'ay',
    'oo', 'ee', 'ii',
    'doo', 'dah', 'dum',
    'sha', 'shoop', 'bop',
    
    # === Lyrics Structure Artifacts ===
    'verse', 'chorus', 'bridge', 'intro', 'outro', 'hook',
    'refrain', 'pre', 'interlude', 'instrumental',
    'lyrics', 'lyric',
    'embed', 'contributor', 'contributors',
    'repeat', 'repeats', 'repeated',
}

print(f"Custom stopwords defined: {len(CUSTOM_STOPWORDS)} words")

Custom stopwords defined: 348 words


## 1. Load Data & Recreate Audio-Based Communities

In [7]:
# Load Spotify dataset
df_tracks = pd.read_csv(DATA_PATH)

# Drop unnamed index column if present
if "Unnamed: 0" in df_tracks.columns:
    df_tracks = df_tracks.drop(columns=["Unnamed: 0"])

print(f"Loaded {len(df_tracks)} tracks")
print(f"Columns: {list(df_tracks.columns)}")

Loaded 73403 tracks
Columns: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


In [8]:
# Audio feature columns (same as notebook 2)
audio_feature_cols = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence'
]

# Clean and aggregate by genre
df_clean = df_tracks.dropna(subset=['track_genre'])
df_clean = df_clean.dropna(subset=audio_feature_cols, how='all')

# Compute genre-level features
grouped = df_clean.groupby('track_genre')
df_genre_feat = grouped[audio_feature_cols].mean()
df_genre_feat['n_tracks'] = grouped.size()
df_genre_feat['avg_popularity'] = grouped['popularity'].mean()

print(f"Number of genres: {len(df_genre_feat)}")
print(f"Total tracks after cleaning: {df_genre_feat['n_tracks'].sum()}")

Number of genres: 114
Total tracks after cleaning: 73403


In [9]:
# Z-score normalization of audio features
scaler = StandardScaler()
X_z = scaler.fit_transform(df_genre_feat[audio_feature_cols])

# Cosine similarity matrix
S_z = cosine_similarity(X_z)
genres_all = df_genre_feat.index.tolist()
n_genres = len(genres_all)

print(f"Computed {n_genres} x {n_genres} similarity matrix")

Computed 114 x 114 similarity matrix


In [10]:
# Build genre similarity graph
G_genre = nx.Graph()

for g in genres_all:
    G_genre.add_node(
        g,
        n_tracks=int(df_genre_feat.loc[g, 'n_tracks']),
        avg_popularity=float(df_genre_feat.loc[g, 'avg_popularity'])
    )

edge_count = 0
for i in range(n_genres):
    for j in range(i + 1, n_genres):
        w = S_z[i, j]
        if w > SIM_THRESHOLD:
            G_genre.add_edge(genres_all[i], genres_all[j], weight=float(w))
            edge_count += 1

print(f"Genre graph: {G_genre.number_of_nodes()} nodes, {edge_count} edges (threshold = {SIM_THRESHOLD})")

Genre graph: 114 nodes, 2058 edges (threshold = 0.3)


In [11]:
# Run Louvain community detection
genre_comms = louvain_communities(G_genre, weight='weight', seed=LOUVAIN_SEED)
Q_audio = modularity(G_genre, genre_comms, weight='weight')

print(f"Found {len(genre_comms)} audio-based communities")
print(f"Modularity (audio): Q = {Q_audio:.4f}")

# Build community mapping: genre -> community_id
comm_map_audio = {}
for cid, comm in enumerate(genre_comms):
    for g in comm:
        comm_map_audio[g] = cid

# Print community sizes
print("\nCommunity sizes:")
for cid, comm in enumerate(genre_comms):
    print(f"  Community {cid}: {len(comm)} genres")

Found 4 audio-based communities
Modularity (audio): Q = 0.4209

Community sizes:
  Community 0: 7 genres
  Community 1: 35 genres
  Community 2: 35 genres
  Community 3: 37 genres


In [12]:
# Store genre info per community for sampling
community_genre_info = {}

for cid, comm in enumerate(genre_comms):
    comm_genres = list(comm)
    comm_df = df_genre_feat.loc[comm_genres].copy()
    community_genre_info[cid] = comm_df

## 2. Genre Filtering & Random Sampling

We apply three filters before sampling:
1. **Non-English genres** - Excluded for fair text comparison
2. **High instrumentalness** - Genres with instrumentalness >= 0.5 likely have few/no lyrics
3. **Low track count** - Genres with < 100 tracks may not be representative

Then we **randomly sample** N genres per community (rather than taking top by popularity).

In [13]:
# ============================================================================
# GENRE FILTERING REPORT
# Show which genres are excluded and why
# ============================================================================

# Collect filtering info
filter_report = []

for genre in df_genre_feat.index:
    n_tracks = df_genre_feat.loc[genre, 'n_tracks']
    instrumentalness = df_genre_feat.loc[genre, 'instrumentalness']
    
    reasons = []
    
    if genre in EXCLUDED_GENRES:
        reasons.append('non-English')
    if instrumentalness >= MAX_INSTRUMENTALNESS:
        reasons.append(f'instrumental ({instrumentalness:.2f})')
    if n_tracks < MIN_TRACKS_PER_GENRE:
        reasons.append(f'low tracks ({int(n_tracks)})')
    
    if reasons:
        filter_report.append({
            'genre': genre,
            'community': comm_map_audio.get(genre, -1),
            'n_tracks': int(n_tracks),
            'instrumentalness': instrumentalness,
            'reasons': ', '.join(reasons)
        })

df_filtered = pd.DataFrame(filter_report)

print("="*80)
print("GENRE FILTERING REPORT")
print("="*80)
print(f"\nTotal genres: {len(df_genre_feat)}")
print(f"Genres excluded: {len(df_filtered)}")
print(f"Genres remaining: {len(df_genre_feat) - len(df_filtered)}")

# Count by reason
non_english_count = sum(1 for r in filter_report if 'non-English' in r['reasons'])
instrumental_count = sum(1 for r in filter_report if 'instrumental' in r['reasons'])
low_tracks_count = sum(1 for r in filter_report if 'low tracks' in r['reasons'])

print(f"\nExclusion reasons (genres may have multiple):")
print(f"  - Non-English: {non_english_count}")
print(f"  - High instrumentalness (>= {MAX_INSTRUMENTALNESS}): {instrumental_count}")
print(f"  - Low track count (< {MIN_TRACKS_PER_GENRE}): {low_tracks_count}")

GENRE FILTERING REPORT

Total genres: 114
Genres excluded: 39
Genres remaining: 75

Exclusion reasons (genres may have multiple):
  - Non-English: 28
  - High instrumentalness (>= 0.5): 12
  - Low track count (< 100): 0


In [14]:
# Show excluded genres by category
print("\n" + "="*80)
print("EXCLUDED GENRES BY CATEGORY")
print("="*80)

# High instrumentalness (likely no lyrics)
instrumental_genres = df_filtered[df_filtered['reasons'].str.contains('instrumental')].sort_values('instrumentalness', ascending=False)
if len(instrumental_genres) > 0:
    print(f"\nHigh Instrumentalness (>= {MAX_INSTRUMENTALNESS}):")
    for _, row in instrumental_genres.iterrows():
        print(f"  {row['genre']}: {row['instrumentalness']:.2f}")

# Low track count
low_track_genres = df_filtered[df_filtered['reasons'].str.contains('low tracks')].sort_values('n_tracks')
if len(low_track_genres) > 0:
    print(f"\nLow Track Count (< {MIN_TRACKS_PER_GENRE}):")
    for _, row in low_track_genres.iterrows():
        print(f"  {row['genre']}: {row['n_tracks']} tracks")


EXCLUDED GENRES BY CATEGORY

High Instrumentalness (>= 0.5):
  study: 0.79
  minimal-techno: 0.74
  sleep: 0.74
  detroit-techno: 0.72
  new-age: 0.71
  ambient: 0.71
  idm: 0.69
  classical: 0.60
  iranian: 0.58
  piano: 0.57
  techno: 0.56
  grindcore: 0.54


In [15]:
def get_eligible_genres(df_genre_feat, community_genres, excluded_genres, 
                        max_instrumentalness, min_tracks):
    """
    Get genres eligible for text analysis after applying all filters.
    
    Returns:
        List of eligible genre names
    """
    eligible = []
    
    for genre in community_genres:
        # Check exclusion list
        if genre in excluded_genres:
            continue
        
        # Check instrumentalness
        if df_genre_feat.loc[genre, 'instrumentalness'] >= max_instrumentalness:
            continue
        
        # Check track count
        if df_genre_feat.loc[genre, 'n_tracks'] < min_tracks:
            continue
        
        eligible.append(genre)
    
    return eligible


def select_random_genres(community_genre_info, df_genre_feat, n_per_community,
                         excluded_genres, max_instrumentalness, min_tracks, seed):
    """
    Randomly select N genres from each community after applying filters.
    If a community has fewer than N eligible genres, take all available.
    
    Returns:
        Dict: {community_id: [list of selected genre names]}
    """
    random.seed(seed)
    selected = {}
    
    for cid, comm_df in community_genre_info.items():
        # Get eligible genres for this community
        eligible = get_eligible_genres(
            df_genre_feat, 
            comm_df.index.tolist(),
            excluded_genres,
            max_instrumentalness,
            min_tracks
        )
        
        # Randomly sample (or take all if fewer than N)
        n_to_take = min(n_per_community, len(eligible))
        selected[cid] = random.sample(eligible, n_to_take) if eligible else []
    
    return selected


# Select random genres from each community
selected_genres = select_random_genres(
    community_genre_info,
    df_genre_feat,
    N_GENRES_PER_COMMUNITY,
    EXCLUDED_GENRES,
    MAX_INSTRUMENTALNESS,
    MIN_TRACKS_PER_GENRE,
    RANDOM_SEED
)

# Report selection
print("="*80)
print("RANDOMLY SELECTED GENRES FOR TEXT ANALYSIS")
print(f"(seed={RANDOM_SEED})")
print("="*80)

total_selected = 0
for cid, genres in selected_genres.items():
    total_eligible = len(get_eligible_genres(
        df_genre_feat,
        community_genre_info[cid].index.tolist(),
        EXCLUDED_GENRES,
        MAX_INSTRUMENTALNESS,
        MIN_TRACKS_PER_GENRE
    ))
    print(f"\nCommunity {cid} ({len(genres)}/{total_eligible} eligible):")
    for g in sorted(genres):
        inst = df_genre_feat.loc[g, 'instrumentalness']
        n = int(df_genre_feat.loc[g, 'n_tracks'])
        print(f"  - {g} (inst={inst:.2f}, n={n})")
    total_selected += len(genres)

print(f"\nTotal genres selected: {total_selected}")

RANDOMLY SELECTED GENRES FOR TEXT ANALYSIS
(seed=42)

Community 0 (2/2 eligible):
  - comedy (inst=0.00, n=947)
  - gospel (inst=0.00, n=789)

Community 1 (10/20 eligible):
  - chill (inst=0.18, n=634)
  - disney (inst=0.38, n=831)
  - indie (inst=0.06, n=318)
  - indie-pop (inst=0.04, n=406)
  - jazz (inst=0.15, n=281)
  - romance (inst=0.05, n=804)
  - sad (inst=0.10, n=623)
  - show-tunes (inst=0.03, n=808)
  - singer-songwriter (inst=0.02, n=267)
  - soul (inst=0.03, n=389)

Community 2 (10/29 eligible):
  - black-metal (inst=0.45, n=880)
  - death-metal (inst=0.29, n=723)
  - deep-house (inst=0.23, n=529)
  - dubstep (inst=0.11, n=435)
  - garage (inst=0.20, n=745)
  - hard-rock (inst=0.05, n=509)
  - hardstyle (inst=0.13, n=785)
  - metalcore (inst=0.04, n=677)
  - power-pop (inst=0.07, n=784)
  - punk (inst=0.04, n=490)

Community 3 (10/24 eligible):
  - afrobeat (inst=0.27, n=835)
  - alternative (inst=0.03, n=193)
  - dance (inst=0.01, n=269)
  - dancehall (inst=0.01, n=701)
 

In [16]:
def sample_random_tracks_from_genre(df, genre, n_tracks, seed):
    """
    Randomly sample tracks from a specific genre.
    
    Args:
        df: Full tracks DataFrame
        genre: Genre name to sample from
        n_tracks: Number of tracks to sample
        seed: Random seed for reproducibility
    
    Returns:
        DataFrame with sampled tracks
    """
    genre_df = df[df['track_genre'] == genre].copy()
    
    # Random sample (or take all if fewer than n_tracks)
    n_available = len(genre_df)
    n_to_sample = min(n_tracks, n_available)
    
    sampled = genre_df.sample(n=n_to_sample, random_state=seed)
    
    return sampled


# Sample tracks for all selected genres
all_samples = []

for cid, genres in selected_genres.items():
    for genre in genres:
        # Use a different seed for each genre to ensure variety
        genre_seed = RANDOM_SEED + hash(genre) % 10000
        sampled = sample_random_tracks_from_genre(df_tracks, genre, SONGS_PER_GENRE, genre_seed)
        sampled = sampled.copy()
        sampled['community_id'] = cid
        all_samples.append(sampled)

df_samples = pd.concat(all_samples, ignore_index=True)

print(f"Total tracks sampled: {len(df_samples)}")
print(f"\nBreakdown by community:")
print(df_samples.groupby('community_id').size())
print(f"\nBreakdown by genre (sample):")
print(df_samples.groupby('track_genre').size().head(10))

Total tracks sampled: 32

Breakdown by community:
community_id
0     2
1    10
2    10
3    10
dtype: int64

Breakdown by genre (sample):
track_genre
afrobeat       1
alternative    1
black-metal    1
chill          1
comedy         1
dance          1
dancehall      1
death-metal    1
deep-house     1
disco          1
dtype: int64


In [17]:
# Save the sample list for reference
sample_path = SAMPLES_DIR / "tracks_to_scrape.csv"
df_samples.to_csv(sample_path, index=False)
print(f"Saved track samples to: {sample_path}")

# Preview
df_samples[['track_name', 'artists', 'track_genre', 'community_id', 'popularity']].head(10)

Saved track samples to: data/text_analysis_output/song_samples/tracks_to_scrape.csv


,track_name,artists,track_genre,community_id,popularity
0,Me Derramar,Ministério Vineyard,gospel,0,46
1,Everyone You Know is Going To Die,Moshe Kasher,comedy,0,20
2,Faaslay,Abdul Hannan,indie-pop,1,50
3,Doses,TheHxliday,sad,1,1
4,Strange (feat. Hillary Smith),Kris Bowers;Hillary Smith,jazz,1,59
5,Верни мне музыку,Сергей Зыков,romance,1,0
6,"Winnie the Pooh - From ""Winnie the Pooh and th...",Disney Studio Chorus,disney,1,21
7,The Yellow House,Matt Dahan;Kelly Lynne D'angelo;Dylan Saunders...,show-tunes,1,22
8,wish u felt the way i do,Finding Hope,chill,1,58
9,Duur,Kamakshi Khanna;OAFF,singer-songwriter,1,37


## 3. Scrape Lyrics from Genius API

In [18]:
def load_env_file(path=".env.local"):
    """
    Load environment variables from a .env file.
    Returns a dict of key-value pairs.
    """
    env_vars = {}
    env_path = Path(path)
    
    if not env_path.exists():
        raise FileNotFoundError(f"{path} not found")
    
    with env_path.open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_vars[key.strip()] = value.strip()
    
    return env_vars


# Load API key
env = load_env_file(ENV_PATH)
GENIUS_ACCESS_TOKEN = env.get("GENIUS_ACCESS_TOKEN")

if not GENIUS_ACCESS_TOKEN:
    raise ValueError(f"GENIUS_ACCESS_TOKEN not found in {ENV_PATH}")

print("Genius API token loaded successfully.")

Genius API token loaded successfully.


In [19]:
# Initialize Genius client
genius = Genius(GENIUS_ACCESS_TOKEN, timeout=15, retries=3)

# Configure Genius client
genius.verbose = False  # Suppress output for cleaner progress
genius.remove_section_headers = True  # Remove [Verse], [Chorus], etc.
genius.skip_non_songs = True  # Skip non-song results
genius.excluded_terms = ["(Remix)", "(Live)"]  # Skip remixes and live versions

print("Genius client initialized.")

Genius client initialized.


In [20]:
def get_primary_artist(artists_str):
    """
    Extract the primary (first) artist from a semicolon-separated string.
    """
    if pd.isna(artists_str):
        return None
    artists = [a.strip() for a in str(artists_str).split(";")]
    return artists[0] if artists else None


def clean_track_name(track_name):
    """
    Clean track name for better Genius search matching.
    Removes common suffixes like (Remastered), (feat. X), etc.
    """
    if pd.isna(track_name):
        return None
    
    name = str(track_name)
    
    # Remove parenthetical suffixes
    patterns = [
        r"\s*\(.*?remaster.*?\)",
        r"\s*\(.*?remix.*?\)",
        r"\s*\(.*?live.*?\)",
        r"\s*\(.*?version.*?\)",
        r"\s*\(.*?edit.*?\)",
        r"\s*\(.*?acoustic.*?\)",
        r"\s*-\s*remaster.*$",
    ]
    
    for pattern in patterns:
        name = re.sub(pattern, "", name, flags=re.IGNORECASE)
    
    return name.strip()


def get_song_language(song):
    """
    Get the language of a Genius song object.
    Returns the language code (e.g., 'en', 'es', 'ko') or None if unavailable.
    """
    try:
        song_dict = song.to_dict()
        return song_dict.get('language', None)
    except Exception:
        return None


def is_english_song(song):
    """
    Check if a Genius song object is in English.
    Returns True if English or unknown, False otherwise.
    """
    lang = get_song_language(song)
    if lang:
        return lang.lower() == 'en'
    # If language not available, assume English (conservative)
    return True


# Test
print(f"Primary artist: {get_primary_artist('Kendrick Lamar;SZA')}")
print(f"Cleaned name: {clean_track_name('Bohemian Rhapsody - 2011 Remaster')}")

Primary artist: Kendrick Lamar
Cleaned name: Bohemian Rhapsody - 2011 Remaster


In [21]:
def scrape_lyrics_for_tracks(df, genius_client, english_only=True, sleep_time=0.5):
    """
    Scrape lyrics for a DataFrame of tracks.
    
    Args:
        df: DataFrame with 'track_name', 'artists', 'track_genre', 'community_id' columns
        genius_client: Initialized Genius API client
        english_only: If True, skip non-English songs
        sleep_time: Seconds to wait between API calls (rate limiting)
    
    Returns:
        List of dicts with lyrics and metadata
    """
    results = []
    total = len(df)
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    print(f"Starting lyrics scraping for {total} tracks...")
    print(f"English only: {english_only}")
    print("-" * 60)
    
    for idx, row in df.iterrows():
        track_name = clean_track_name(row['track_name'])
        artist = get_primary_artist(row['artists'])
        genre = row['track_genre']
        community_id = row['community_id']
        track_id = row.get('track_id', None)
        
        if not track_name or not artist:
            fail_count += 1
            continue
        
        try:
            # Search for song
            song = genius_client.search_song(track_name, artist)
            
            if song is None:
                fail_count += 1
                continue
            
            # Get language
            language = get_song_language(song)
            
            # Language check (if enabled)
            if english_only and not is_english_song(song):
                skip_count += 1
                continue
            
            # Get lyrics
            lyrics = song.lyrics
            
            if lyrics and len(lyrics) > 50:  # Minimal lyrics threshold
                results.append({
                    'track_id': track_id,
                    'track_name': row['track_name'],
                    'artist': artist,
                    'genre': genre,
                    'community_id': community_id,
                    'genius_title': song.title,
                    'genius_artist': song.artist,
                    'language': language,
                    'lyrics': lyrics
                })
                success_count += 1
            else:
                fail_count += 1
                
        except Exception as e:
            fail_count += 1
        
        # Progress update every 25 tracks
        processed = success_count + skip_count + fail_count
        if processed % 25 == 0:
            print(f"  Progress: {processed}/{total} | Success: {success_count} | Skipped: {skip_count} | Failed: {fail_count}")
        
        # Rate limiting
        time.sleep(sleep_time)
    
    print("-" * 60)
    print(f"Scraping complete!")
    print(f"  Total processed: {total}")
    print(f"  Successful: {success_count} ({100*success_count/total:.1f}%)")
    print(f"  Skipped (non-English): {skip_count}")
    print(f"  Failed: {fail_count}")
    
    return results

In [22]:
# ============================================================================
# MAIN SCRAPING CELL
# This will take a while depending on SONGS_PER_GENRE and number of genres.
# Progress is saved incrementally, so you can interrupt and resume.
# ============================================================================

# Check if we have previously saved results
checkpoint_path = OUTPUT_DIR / "lyrics_checkpoint.json"

if checkpoint_path.exists():
    print(f"Found checkpoint at {checkpoint_path}")
    print("Loading previous results...")
    with open(checkpoint_path, 'r', encoding='utf-8') as f:
        scraped_lyrics = json.load(f)
    print(f"Loaded {len(scraped_lyrics)} previously scraped tracks.")
else:
    print("No checkpoint found. Starting fresh scrape...")
    scraped_lyrics = []

# Identify which tracks still need scraping
if scraped_lyrics:
    scraped_ids = {r['track_id'] for r in scraped_lyrics if r.get('track_id')}
    df_remaining = df_samples[~df_samples['track_id'].isin(scraped_ids)]
    print(f"Remaining tracks to scrape: {len(df_remaining)}")
else:
    df_remaining = df_samples

# Run scraping (only if there are remaining tracks)
if len(df_remaining) > 0:
    new_results = scrape_lyrics_for_tracks(
        df_remaining, 
        genius, 
        english_only=ENGLISH_ONLY,
        sleep_time=0.5
    )
    
    # Merge with previous results
    scraped_lyrics.extend(new_results)
    
    # Save checkpoint
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump(scraped_lyrics, f, ensure_ascii=False, indent=2)
    print(f"\nCheckpoint saved to {checkpoint_path}")
else:
    print("All tracks already scraped!")

print(f"\nTotal lyrics collected: {len(scraped_lyrics)}")

Found checkpoint at data/text_analysis_output/lyrics_checkpoint.json
Loading previous results...
Loaded 0 previously scraped tracks.
Starting lyrics scraping for 32 tracks...
English only: True
------------------------------------------------------------
  Progress: 25/32 | Success: 0 | Skipped: 0 | Failed: 25
------------------------------------------------------------
Scraping complete!
  Total processed: 32
  Successful: 0 (0.0%)
  Skipped (non-English): 0
  Failed: 32

Checkpoint saved to data/text_analysis_output/lyrics_checkpoint.json

Total lyrics collected: 0


In [23]:
# Convert to DataFrame for analysis
df_lyrics = pd.DataFrame(scraped_lyrics)

print(f"Lyrics DataFrame shape: {df_lyrics.shape}")
print(f"\nLyrics by community:")
print(df_lyrics.groupby('community_id').size())
print(f"\nLyrics by genre (top 10):")
print(df_lyrics.groupby('genre').size().sort_values(ascending=False).head(10))

Lyrics DataFrame shape: (0, 0)

Lyrics by community:


KeyError: 'community_id'

In [ ]:
# Save lyrics to separate files per community
for cid in df_lyrics['community_id'].unique():
    comm_lyrics = df_lyrics[df_lyrics['community_id'] == cid]
    
    # Save as JSON
    json_path = LYRICS_DIR / f"community_{cid}_lyrics.json"
    comm_lyrics.to_json(json_path, orient='records', force_ascii=False, indent=2)
    
    # Save as CSV (lyrics truncated for readability)
    csv_df = comm_lyrics.copy()
    csv_df['lyrics_preview'] = csv_df['lyrics'].str[:200] + '...'
    csv_df = csv_df.drop(columns=['lyrics'])
    csv_path = SAMPLES_DIR / f"community_{cid}_songs.csv"
    csv_df.to_csv(csv_path, index=False)
    
    print(f"Community {cid}: Saved {len(comm_lyrics)} lyrics to {json_path}")

# Also save full lyrics DataFrame
full_path = OUTPUT_DIR / "all_lyrics.csv"
df_lyrics.to_csv(full_path, index=False)
print(f"\nSaved all lyrics to {full_path}")

## 4. Text Preprocessing

In [ ]:
import string
from collections import Counter

# NLTK for stopwords and lemmatization
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    NLTK_AVAILABLE = True
    NLTK_STOPWORDS = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
except ImportError:
    print("NLTK not available. Using basic preprocessing.")
    NLTK_AVAILABLE = False
    NLTK_STOPWORDS = set()

# Combine NLTK stopwords with our custom lyrics stopwords
ALL_STOPWORDS = NLTK_STOPWORDS | CUSTOM_STOPWORDS

# Add common themes if configured for removal
THEMES_TO_REMOVE = {word for word, remove in REMOVE_COMMON_THEMES.items() if remove}
ALL_STOPWORDS = ALL_STOPWORDS | THEMES_TO_REMOVE

print(f"NLTK available: {NLTK_AVAILABLE}")
print(f"NLTK stopwords: {len(NLTK_STOPWORDS)}")
print(f"Custom stopwords: {len(CUSTOM_STOPWORDS)}")
print(f"Common themes removed: {THEMES_TO_REMOVE if THEMES_TO_REMOVE else 'None'}")
print(f"Total stopwords: {len(ALL_STOPWORDS)}")

In [ ]:
def preprocess_lyrics(text, lemmatize=True, min_word_length=2):
    """
    Preprocess lyrics text for TF-IDF analysis.
    
    Steps:
    1. Lowercase
    2. Remove section headers and Genius artifacts
    3. Remove punctuation and numbers
    4. Tokenize
    5. Remove stopwords (NLTK + custom)
    6. Optional lemmatization
    7. Filter short tokens
    
    Returns:
        Cleaned text as a single string (for TfidfVectorizer)
    """
    if pd.isna(text) or not text:
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove section headers like [Verse 1], [Chorus], etc.
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove contributor/embed notices (Genius artifacts)
    text = re.sub(r'\d+\s*contributors?', '', text)
    text = re.sub(r'embed$', '', text)
    text = re.sub(r'you might also like', '', text)
    
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Tokenize
    tokens = text.split()
    
    # Remove all stopwords (NLTK + custom + themes)
    tokens = [t for t in tokens if t not in ALL_STOPWORDS]
    
    # Filter short tokens
    tokens = [t for t in tokens if len(t) >= min_word_length]
    
    # Lemmatization (if NLTK available and requested)
    if lemmatize and NLTK_AVAILABLE:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return ' '.join(tokens)


# Test preprocessing
sample_lyrics = """
[Verse 1]
I'm walking down the street, yeah
Gonna think about you, oh oh oh
Baby, I wanna tell you something
[Chorus]
Love is all we need!
Never gonna let you go
123Embed
"""

print("Original:")
print(sample_lyrics)
print("\nPreprocessed:")
print(preprocess_lyrics(sample_lyrics))

In [ ]:
# Apply preprocessing to all lyrics
df_lyrics['lyrics_clean'] = df_lyrics['lyrics'].apply(preprocess_lyrics)

# Check for empty results
empty_count = (df_lyrics['lyrics_clean'].str.len() == 0).sum()
print(f"Tracks with empty lyrics after preprocessing: {empty_count}")

# Remove empty entries
df_lyrics = df_lyrics[df_lyrics['lyrics_clean'].str.len() > 0]
print(f"Remaining tracks: {len(df_lyrics)}")

# Preview
print("\nSample preprocessed lyrics (first 200 chars):")
for i in range(min(3, len(df_lyrics))):
    row = df_lyrics.iloc[i]
    print(f"\n{row['track_name']} ({row['genre']}):")
    print(f"  {row['lyrics_clean'][:200]}...")

## 5. TF-IDF Representation

In [ ]:
# Compute TF-IDF at the track level
# Using UNIGRAMS ONLY to avoid duplicate words in word clouds
tfidf_vectorizer = TfidfVectorizer(
    min_df=5,           # Ignore terms appearing in fewer than 5 documents
    max_df=0.5,         # Ignore terms appearing in more than 50% of documents
    ngram_range=(1, 1), # UNIGRAMS ONLY - avoids "love", "love you", "my love" duplicates
    max_features=5000   # Limit vocabulary size
)

# Fit and transform
tfidf_matrix = tfidf_vectorizer.fit_transform(df_lyrics['lyrics_clean'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  - {tfidf_matrix.shape[0]} tracks")
print(f"  - {tfidf_matrix.shape[1]} unique words (unigrams only)")

# Get feature names
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nSample terms: {list(feature_names[:20])}")

In [ ]:
# Aggregate TF-IDF to genre level (mean of track vectors per genre)
genres_with_lyrics = df_lyrics['genre'].unique()
n_terms = tfidf_matrix.shape[1]

# Create genre-level TF-IDF matrix
genre_tfidf = np.zeros((len(genres_with_lyrics), n_terms))
genre_track_counts = {}

for i, genre in enumerate(genres_with_lyrics):
    genre_mask = df_lyrics['genre'] == genre
    genre_vectors = tfidf_matrix[genre_mask.values]
    
    # Mean TF-IDF vector for genre
    genre_tfidf[i] = genre_vectors.mean(axis=0).A1
    genre_track_counts[genre] = genre_mask.sum()

print(f"Genre-level TF-IDF matrix shape: {genre_tfidf.shape}")
print(f"\nTracks per genre:")
for genre, count in sorted(genre_track_counts.items(), key=lambda x: -x[1]):
    print(f"  {genre}: {count}")

In [ ]:
# Helper functions for getting top terms
def get_top_terms_for_genre(genre_idx, top_n=10):
    """
    Get the top N terms by TF-IDF weight for a genre.
    """
    genre_vector = genre_tfidf[genre_idx]
    top_indices = genre_vector.argsort()[-top_n:][::-1]
    return [(feature_names[idx], genre_vector[idx]) for idx in top_indices]


def get_top_terms_for_genre_by_name(genre_name, top_n=10):
    """
    Get the top N terms by TF-IDF weight for a genre by name.
    """
    if genre_name not in genres_with_lyrics:
        return []
    idx = list(genres_with_lyrics).index(genre_name)
    return get_top_terms_for_genre(idx, top_n)


# Show top terms for a few genres
print("Top TF-IDF terms per genre:")
print("=" * 60)

for i, genre in enumerate(genres_with_lyrics):
    top_terms = get_top_terms_for_genre(i, top_n=8)
    terms_str = ", ".join([f"{t[0]} ({t[1]:.3f})" for t in top_terms])
    print(f"\n{genre}:")
    print(f"  {terms_str}")

---

## 6. Word Clouds

We use word clouds to **semantically interpret** the audio communities discovered in notebook 2. The key question: *Do genres that sound similar also talk about similar things?*

We generate:
1. **Community-level word clouds** - What themes characterize each audio cluster?
2. **Genre-level word clouds** - How do archetypal genres compare thematically?

### 6.1 Helper Functions

In [ ]:
# Create community color palette (will be reused for word clouds)
n_communities = len(genre_comms)
COMMUNITY_COLORS = sns.color_palette('tab10', n_colors=n_communities)
COMMUNITY_COLOR_MAP = {cid: COMMUNITY_COLORS[cid] for cid in range(n_communities)}

# Convert to hex for word clouds
def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))

COMMUNITY_COLOR_HEX = {cid: rgb_to_hex(color) for cid, color in COMMUNITY_COLOR_MAP.items()}

print("Community colors (hex):")
for cid, hex_color in COMMUNITY_COLOR_HEX.items():
    print(f"  Community {cid}: {hex_color}")

In [ ]:
def get_community_tfidf_aggregated(df_lyrics_subset, tfidf_mat, community_id):
    """
    Aggregate TF-IDF vectors for all tracks in a community.
    Returns mean TF-IDF vector for the community.
    """
    comm_mask = df_lyrics_subset['community_id'] == community_id
    comm_vectors = tfidf_mat[comm_mask.values]
    
    if comm_vectors.shape[0] == 0:
        return None
    
    return comm_vectors.mean(axis=0).A1


def get_top_terms_from_vector(tfidf_vector, feat_names, top_n=10):
    """
    Get top N terms from a TF-IDF vector.
    Returns dict: {term: weight}
    """
    top_indices = tfidf_vector.argsort()[-top_n:][::-1]
    return {feat_names[idx]: tfidf_vector[idx] for idx in top_indices}


def create_wordcloud(word_weights, color, max_words=50):
    """
    Create a word cloud from word weights dict.
    
    Args:
        word_weights: dict of {word: weight}
        color: hex color string for the word cloud
        max_words: maximum number of words to display
    
    Returns:
        WordCloud object
    """
    def color_func(*args, **kwargs):
        return color
    
    wc = WordCloud(
        width=800,
        height=400,
        background_color='white',
        max_words=max_words,
        color_func=color_func,
        prefer_horizontal=0.7,
        min_font_size=10
    ).generate_from_frequencies(word_weights)
    
    return wc


print("Word cloud helper functions defined.")

### 6.2 Community Word Clouds

These word clouds show the **most distinctive terms** for each audio community, revealing the thematic "personality" of genres that sound similar.

In [ ]:
# Get unique communities in the lyrics data
communities_in_data = sorted(df_lyrics['community_id'].unique())
n_comms_wc = len(communities_in_data)

print(f"Generating word clouds for {n_comms_wc} communities...")
print(f"\nTracks per community:")
for cid in communities_in_data:
    n_tracks = (df_lyrics['community_id'] == cid).sum()
    n_genres = df_lyrics[df_lyrics['community_id'] == cid]['genre'].nunique()
    print(f"  Community {cid}: {n_tracks} tracks across {n_genres} genres")

In [ ]:
# Generate community word clouds
fig, axes = plt.subplots(1, n_comms_wc, figsize=(5 * n_comms_wc, 5))

# Handle case of single community
if n_comms_wc == 1:
    axes = [axes]

for i, cid in enumerate(communities_in_data):
    # Get aggregated TF-IDF for community
    comm_vector = get_community_tfidf_aggregated(df_lyrics, tfidf_matrix, cid)
    
    if comm_vector is None:
        axes[i].text(0.5, 0.5, 'No data', ha='center', va='center')
        axes[i].set_title(f'Community {cid}')
        axes[i].axis('off')
        continue
    
    # Get top terms
    top_terms = get_top_terms_from_vector(comm_vector, feature_names, top_n=WORDCLOUD_MAX_WORDS)
    
    # Get community color
    color = COMMUNITY_COLOR_HEX.get(cid, '#333333')
    
    # Create word cloud
    wc = create_wordcloud(top_terms, color, max_words=WORDCLOUD_MAX_WORDS)
    
    # Plot
    axes[i].imshow(wc, interpolation='bilinear')
    n_tracks = (df_lyrics['community_id'] == cid).sum()
    n_genres = df_lyrics[df_lyrics['community_id'] == cid]['genre'].nunique()
    axes[i].set_title(f'Community {cid}\n({n_genres} genres, {n_tracks} tracks)', fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Thematic Word Clouds by Audio Community', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'wordclouds_community.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved to {OUTPUT_DIR / 'wordclouds_community.png'}")

In [ ]:
# Print top terms per community for reference
print("\n" + "="*60)
print("TOP TERMS PER AUDIO COMMUNITY")
print("="*60)

for cid in communities_in_data:
    comm_vector = get_community_tfidf_aggregated(df_lyrics, tfidf_matrix, cid)
    if comm_vector is None:
        continue
    
    top_terms = get_top_terms_from_vector(comm_vector, feature_names, top_n=15)
    terms_str = ", ".join([f"{term}" for term in top_terms.keys()])
    
    # Get genres in this community
    genres_in_comm = df_lyrics[df_lyrics['community_id'] == cid]['genre'].unique()
    
    print(f"\nCommunity {cid}:")
    print(f"  Genres: {', '.join(sorted(genres_in_comm))}")
    print(f"  Top terms: {terms_str}")

### 6.3 Genre Word Clouds (Archetypal Genres)

These word clouds show the thematic vocabulary of well-known genres, allowing intuitive cross-genre comparisons.

In [ ]:
# Check which archetypal genres have lyrics data
available_archetypal = [g for g in ARCHETYPAL_GENRES if g in genres_with_lyrics]
missing_archetypal = [g for g in ARCHETYPAL_GENRES if g not in genres_with_lyrics]

print(f"Archetypal genres with lyrics data: {len(available_archetypal)}/{len(ARCHETYPAL_GENRES)}")
if missing_archetypal:
    print(f"Missing: {missing_archetypal}")

# Show track counts for available genres
print(f"\nTrack counts:")
for genre in available_archetypal:
    count = genre_track_counts.get(genre, 0)
    comm_id = comm_map_audio.get(genre, '?')
    print(f"  {genre}: {count} tracks (Community {comm_id})")

In [ ]:
# Generate genre word clouds (only if we have archetypal genres with data)
if available_archetypal:
    # Arrange in a grid
    n_genres_plot = len(available_archetypal)
    n_cols = 6
    n_rows = (n_genres_plot + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_genres_plot == 1 else axes
    
    for i, genre in enumerate(available_archetypal):
        ax = axes[i]
        
        # Get top terms for genre
        top_terms = get_top_terms_for_genre_by_name(genre, top_n=WORDCLOUD_MAX_WORDS)
        
        if top_terms:
            word_weights = {term: weight for term, weight in top_terms}
            
            # Get community color for this genre
            comm_id = comm_map_audio.get(genre, 0)
            color = COMMUNITY_COLOR_HEX.get(comm_id, '#333333')
            
            wc = create_wordcloud(word_weights, color, max_words=WORDCLOUD_MAX_WORDS)
            
            ax.imshow(wc, interpolation='bilinear')
            n_tracks = genre_track_counts.get(genre, 0)
            ax.set_title(f'{genre}\n(C{comm_id}, n={n_tracks})', fontsize=11, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.set_title(genre, fontsize=11)
        
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n_genres_plot, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Word Clouds for Archetypal Genres\n(Color = Audio Community)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'wordclouds_genre_archetypal.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved to {OUTPUT_DIR / 'wordclouds_genre_archetypal.png'}")
else:
    print("No archetypal genres have lyrics data. Skipping genre word clouds.")

---

## 7. Sentiment Analysis & Additional Visualizations

Beyond word clouds, we analyze the **emotional tone** of lyrics using VADER sentiment analysis. This reveals whether audio communities have distinct emotional "personalities" (e.g., is rock more angry? is pop more positive?).

### 7.1 Sentiment Analysis with VADER

In [ ]:
# Install/import VADER
try:
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    nltk.download('vader_lexicon', quiet=True)
    VADER_AVAILABLE = True
except ImportError:
    print("VADER not available. Skipping sentiment analysis.")
    VADER_AVAILABLE = False

if VADER_AVAILABLE:
    # Initialize VADER
    sia = SentimentIntensityAnalyzer()
    print("VADER sentiment analyzer initialized.")
    
    # Test on sample text
    test_texts = [
        "I love you so much, you make me happy!",
        "I hate everything, this world is terrible.",
        "The sky is blue and the grass is green."
    ]
    print("\nTest sentiment scores:")
    for text in test_texts:
        scores = sia.polarity_scores(text)
        print(f"  '{text[:40]}...' -> compound: {scores['compound']:.3f}")

In [ ]:
if VADER_AVAILABLE:
    # Compute sentiment for all lyrics (using original lyrics, not preprocessed)
    def get_sentiment(text):
        """Get VADER compound sentiment score for text."""
        if pd.isna(text) or not text:
            return 0.0
        scores = sia.polarity_scores(text)
        return scores['compound']
    
    # Apply to all lyrics
    df_lyrics['sentiment'] = df_lyrics['lyrics'].apply(get_sentiment)
    
    print(f"Sentiment scores computed for {len(df_lyrics)} tracks.")
    print(f"\nSentiment distribution:")
    print(f"  Mean: {df_lyrics['sentiment'].mean():.3f}")
    print(f"  Std:  {df_lyrics['sentiment'].std():.3f}")
    print(f"  Min:  {df_lyrics['sentiment'].min():.3f}")
    print(f"  Max:  {df_lyrics['sentiment'].max():.3f}")
    
    # Show examples of positive and negative songs
    print("\nMost positive songs:")
    for _, row in df_lyrics.nlargest(3, 'sentiment').iterrows():
        print(f"  {row['sentiment']:.3f} - {row['track_name']} ({row['genre']})")
    
    print("\nMost negative songs:")
    for _, row in df_lyrics.nsmallest(3, 'sentiment').iterrows():
        print(f"  {row['sentiment']:.3f} - {row['track_name']} ({row['genre']})")

### 7.2 Sentiment by Community (Box Plot)

In [ ]:
if VADER_AVAILABLE:
    # Create box plot of sentiment by community
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Prepare data for plotting
    communities_sorted = sorted(df_lyrics['community_id'].unique())
    
    # Create box plot with community colors
    bp = ax.boxplot(
        [df_lyrics[df_lyrics['community_id'] == cid]['sentiment'].values for cid in communities_sorted],
        labels=[f'C{cid}' for cid in communities_sorted],
        patch_artist=True
    )
    
    # Color boxes by community
    for i, cid in enumerate(communities_sorted):
        bp['boxes'][i].set_facecolor(COMMUNITY_COLOR_MAP[cid])
        bp['boxes'][i].set_alpha(0.7)
    
    # Add horizontal line at 0 (neutral)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    
    # Labels
    ax.set_xlabel('Audio Community', fontsize=12)
    ax.set_ylabel('Sentiment Score (VADER Compound)', fontsize=12)
    ax.set_title('Lyrical Sentiment Distribution by Audio Community', fontsize=14, fontweight='bold')
    
    # Add text annotation for interpretation
    ax.text(0.02, 0.98, 'Positive ↑', transform=ax.transAxes, fontsize=10, 
            verticalalignment='top', color='green', alpha=0.7)
    ax.text(0.02, 0.02, 'Negative ↓', transform=ax.transAxes, fontsize=10, 
            verticalalignment='bottom', color='red', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sentiment_by_community.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved to {OUTPUT_DIR / 'sentiment_by_community.png'}")
    
    # Print summary stats
    print("\nSentiment by community (mean):")
    for cid in communities_sorted:
        comm_sentiment = df_lyrics[df_lyrics['community_id'] == cid]['sentiment']
        genres_in_comm = df_lyrics[df_lyrics['community_id'] == cid]['genre'].unique()
        print(f"  Community {cid}: mean={comm_sentiment.mean():.3f}, std={comm_sentiment.std():.3f}")
        print(f"    Genres: {', '.join(sorted(genres_in_comm)[:5])}{'...' if len(genres_in_comm) > 5 else ''}")

### 7.3 Sentiment by Archetypal Genre (Bar Chart)

In [ ]:
if VADER_AVAILABLE and available_archetypal:
    # Compute mean sentiment for each archetypal genre
    genre_sentiments = []
    
    for genre in available_archetypal:
        genre_df = df_lyrics[df_lyrics['genre'] == genre]
        if len(genre_df) > 0:
            genre_sentiments.append({
                'genre': genre,
                'sentiment_mean': genre_df['sentiment'].mean(),
                'sentiment_std': genre_df['sentiment'].std(),
                'n_tracks': len(genre_df),
                'community_id': comm_map_audio.get(genre, 0)
            })
    
    df_genre_sentiment = pd.DataFrame(genre_sentiments)
    df_genre_sentiment = df_genre_sentiment.sort_values('sentiment_mean', ascending=True)
    
    # Create horizontal bar chart
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Bar colors by community
    bar_colors = [COMMUNITY_COLOR_MAP[cid] for cid in df_genre_sentiment['community_id']]
    
    bars = ax.barh(
        df_genre_sentiment['genre'],
        df_genre_sentiment['sentiment_mean'],
        xerr=df_genre_sentiment['sentiment_std'],
        color=bar_colors,
        alpha=0.8,
        capsize=3
    )
    
    # Add vertical line at 0
    ax.axvline(x=0, color='gray', linestyle='--', linewidth=1)
    
    # Labels
    ax.set_xlabel('Mean Sentiment Score (VADER Compound)', fontsize=12)
    ax.set_ylabel('Genre', fontsize=12)
    ax.set_title('Lyrical Sentiment by Archetypal Genre\n(Color = Audio Community)', fontsize=14, fontweight='bold')
    
    # Add value labels on bars
    for i, (idx, row) in enumerate(df_genre_sentiment.iterrows()):
        x_pos = row['sentiment_mean']
        ha = 'left' if x_pos >= 0 else 'right'
        offset = 0.02 if x_pos >= 0 else -0.02
        ax.text(x_pos + offset, i, f"{x_pos:.2f}", va='center', ha=ha, fontsize=9)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sentiment_by_genre.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved to {OUTPUT_DIR / 'sentiment_by_genre.png'}")

### 7.4 Top Words Comparison (Bar Chart)

In [ ]:
# Create a side-by-side bar chart showing top 10 words per community
TOP_N_WORDS = 10

fig, axes = plt.subplots(1, n_comms_wc, figsize=(5 * n_comms_wc, 6), sharey=False)

if n_comms_wc == 1:
    axes = [axes]

for i, cid in enumerate(communities_in_data):
    ax = axes[i]
    
    # Get aggregated TF-IDF for community
    comm_vector = get_community_tfidf_aggregated(df_lyrics, tfidf_matrix, cid)
    
    if comm_vector is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(f'Community {cid}')
        continue
    
    # Get top terms
    top_terms = get_top_terms_from_vector(comm_vector, feature_names, top_n=TOP_N_WORDS)
    
    # Create horizontal bar chart
    words = list(top_terms.keys())
    weights = list(top_terms.values())
    
    # Reverse for top-to-bottom ordering
    words = words[::-1]
    weights = weights[::-1]
    
    color = COMMUNITY_COLOR_MAP[cid]
    ax.barh(words, weights, color=color, alpha=0.8)
    
    # Get genres in community for subtitle
    genres_in_comm = df_lyrics[df_lyrics['community_id'] == cid]['genre'].unique()
    genre_str = ', '.join(sorted(genres_in_comm)[:3]) + ('...' if len(genres_in_comm) > 3 else '')
    
    ax.set_xlabel('TF-IDF Weight', fontsize=10)
    ax.set_title(f'Community {cid}\n({genre_str})', fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=10)

plt.suptitle(f'Top {TOP_N_WORDS} Words by Audio Community', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top_words_by_community.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved to {OUTPUT_DIR / 'top_words_by_community.png'}")

---

## 8. Summary

In [ ]:
print("="*80)
print("TEXT ANALYSIS SUMMARY")
print("="*80)

print(f"\n CONFIGURATION:")
print(f"   - Random seed: {RANDOM_SEED}")
print(f"   - Genres per community: {N_GENRES_PER_COMMUNITY} (random)")
print(f"   - Songs per genre: {SONGS_PER_GENRE} (random)")
print(f"   - Min tracks threshold: {MIN_TRACKS_PER_GENRE}")
print(f"   - Max instrumentalness: {MAX_INSTRUMENTALNESS}")

print(f"\n DATA:")
print(f"   - Total genres in dataset: {len(genres_all)}")
print(f"   - Genres excluded (all filters): {len(df_filtered)}")
print(f"   - Genres sampled for lyrics: {len(genres_with_lyrics)}")
print(f"   - Total tracks with lyrics: {len(df_lyrics)}")
print(f"   - TF-IDF vocabulary size: {len(feature_names)} (unigrams only)")

print(f"\n STOPWORDS:")
print(f"   - NLTK stopwords: {len(NLTK_STOPWORDS)}")
print(f"   - Custom stopwords: {len(CUSTOM_STOPWORDS)}")
print(f"   - Common themes removed: {THEMES_TO_REMOVE if THEMES_TO_REMOVE else 'None'}")
print(f"   - Total stopwords: {len(ALL_STOPWORDS)}")

print(f"\n AUDIO COMMUNITIES:")
print(f"   - Number of communities: {len(genre_comms)}")
print(f"   - Modularity Q: {Q_audio:.4f}")

print(f"\n WORD CLOUDS:")
print(f"   - Community word clouds: {n_comms_wc}")
print(f"   - Archetypal genre word clouds: {len(available_archetypal)}")

if VADER_AVAILABLE:
    print(f"\n SENTIMENT ANALYSIS:")
    print(f"   - Overall mean sentiment: {df_lyrics['sentiment'].mean():.3f}")
    print(f"   - Sentiment range: [{df_lyrics['sentiment'].min():.3f}, {df_lyrics['sentiment'].max():.3f}]")

print(f"\n OUTPUTS:")
print(f"   - Community word clouds: {OUTPUT_DIR / 'wordclouds_community.png'}")
print(f"   - Genre word clouds: {OUTPUT_DIR / 'wordclouds_genre_archetypal.png'}")
print(f"   - Sentiment by community: {OUTPUT_DIR / 'sentiment_by_community.png'}")
print(f"   - Sentiment by genre: {OUTPUT_DIR / 'sentiment_by_genre.png'}")
print(f"   - Top words comparison: {OUTPUT_DIR / 'top_words_by_community.png'}")
print(f"   - Lyrics data: {OUTPUT_DIR / 'all_lyrics.csv'}")
print(f"   - Top terms: {OUTPUT_DIR / 'genre_top_tfidf_terms.csv'}")

In [ ]:
# Save top terms per genre for reference
top_terms_df = []
for i, genre in enumerate(genres_with_lyrics):
    top_terms = get_top_terms_for_genre(i, top_n=15)
    comm_id = comm_map_audio.get(genre, -1)
    for term, weight in top_terms:
        top_terms_df.append({
            'genre': genre,
            'community_id': comm_id,
            'term': term,
            'tfidf_weight': weight
        })

pd.DataFrame(top_terms_df).to_csv(OUTPUT_DIR / 'genre_top_tfidf_terms.csv', index=False)
print(f"Saved top terms per genre to {OUTPUT_DIR / 'genre_top_tfidf_terms.csv'}")

In [ ]:
# Save sentiment data if available
if VADER_AVAILABLE:
    # Save full lyrics with sentiment
    df_lyrics.to_csv(OUTPUT_DIR / 'all_lyrics_with_sentiment.csv', index=False)
    print(f"Saved lyrics with sentiment to {OUTPUT_DIR / 'all_lyrics_with_sentiment.csv'}")
    
    # Save genre-level sentiment summary
    genre_sentiment_summary = df_lyrics.groupby('genre').agg({
        'sentiment': ['mean', 'std', 'min', 'max', 'count'],
        'community_id': 'first'
    }).round(3)
    genre_sentiment_summary.columns = ['sentiment_mean', 'sentiment_std', 'sentiment_min', 
                                        'sentiment_max', 'n_tracks', 'community_id']
    genre_sentiment_summary = genre_sentiment_summary.sort_values('sentiment_mean', ascending=False)
    genre_sentiment_summary.to_csv(OUTPUT_DIR / 'genre_sentiment_summary.csv')
    print(f"Saved genre sentiment summary to {OUTPUT_DIR / 'genre_sentiment_summary.csv'}")